# Mi vista previa de cobertura de viajes

Este análisis usa solamente el snapshot existente 2026-09-29.
Una futura fila Gold representará un viaje declarado dentro de su versión y hash.
La consulta solo lee tres tablas Silver; no crea vistas, tablas ni auditorías.
Verifiqué esta consulta mediante Genie y recuperé cuatro grupos completos.
No registré una ejecución completa del notebook en Workspace.
No necesito ejecutarlo antes de cada corrida de producción.


In [ ]:
%sql
with selected_trips as (
  select source_feed_version, source_sha256, trip_id
  from workspace.gtfs_bogota.silver_gtfs_trips
  where source_feed_version = '2026-09-29'
    and source_sha256 = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
), visits as (
  select source_feed_version, source_sha256, trip_id,
         count(*) as visit_count,
         sum(case when arrival_time_seconds is not null
                       and departure_time_seconds is not null then 1 else 0 end) as complete_time_pair_count
  from workspace.gtfs_bogota.silver_gtfs_stop_times
  where source_feed_version = '2026-09-29'
    and source_sha256 = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
  group by source_feed_version, source_sha256, trip_id
), frequency_context as (
  select source_feed_version, source_sha256, trip_id,
         count(*) as frequency_window_count,
         min(exact_times_effective) as minimum_exact_times,
         max(exact_times_effective) as maximum_exact_times
  from workspace.gtfs_bogota.silver_gtfs_frequencies
  where source_feed_version = '2026-09-29'
    and source_sha256 = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
  group by source_feed_version, source_sha256, trip_id
), coverage as (
  select t.source_feed_version, t.source_sha256, t.trip_id,
         coalesce(v.visit_count, 0) as visit_count,
         coalesce(v.complete_time_pair_count, 0) as complete_time_pair_count,
         coalesce(f.frequency_window_count, 0) as frequency_window_count,
         case when v.visit_count is null then 'sin_visitas'
              when v.visit_count = 1 then 'una_visita'
              else 'varias_visitas' end as coverage_status,
         case when f.trip_id is null then 'sin_frecuencias'
              when f.minimum_exact_times = 0 and f.maximum_exact_times = 0 then 'por_frecuencia'
              when f.minimum_exact_times = 1 and f.maximum_exact_times = 1 then 'horario_exacto_por_frecuencia'
              else 'contexto_inconsistente' end as scheduling_context
  from selected_trips t
  left join visits v
    on t.source_feed_version = v.source_feed_version
   and t.source_sha256 = v.source_sha256 and t.trip_id = v.trip_id
  left join frequency_context f
    on t.source_feed_version = f.source_feed_version
   and t.source_sha256 = f.source_sha256 and t.trip_id = f.trip_id
)
select source_feed_version, source_sha256, coverage_status, scheduling_context,
       count(*) as declared_trips,
       sum(visit_count) as published_visits,
       sum(frequency_window_count) as frequency_windows,
       sum(visit_count - complete_time_pair_count) as visits_without_complete_time_pair,
       sum(case when visit_count >= 2 then 1 else 0 end) as trips_with_sequence,
       sum(case when visit_count > 0 and complete_time_pair_count = visit_count
                then 1 else 0 end) as trips_with_all_time_pairs
from coverage
group by source_feed_version, source_sha256, coverage_status, scheduling_context
order by coverage_status, scheduling_context


## Cómo interpreto el resultado

Sin visitas: trips declara el viaje, pero stop_times no aporta visitas.
Una visita no permite describir un recorrido entre dos paradas.
Varias visitas no demuestran que el recorrido esté completo.
El contexto de frecuencias es independiente de la cantidad de visitas.
Sin frequencies no significa que todas las horas publicadas sean exactas.
Los pares de horas completos indican presencia, no exactitud temporal ni operación observada.
No multiplico visitas por ventanas ni por fechas del calendario.
El SQL usa los nombres reales de mi entorno; no constituye todavía un publicador configurable.
La futura implementación Gold recibirá catálogo, schema y snapshot por configuración.
